# Notebook 05 — Biological Interpretation of Topological Features

**Goal:** Bridge the gap between topological signatures (H1 cycles, Mapper nodes) and biological meaning.

This notebook implements the full interpretation protocol:
1. Map persistent H1 cycles to gene sets
2. Run GO/KEGG enrichment on cycle-associated genes
3. Analyze Mapper nodes enriched in resilient individuals
4. Cross-reference with longevity databases (GenAge, LongevityMap)
5. Visualize enriched subnetworks

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from data_utils import generate_synthetic_multimics, preprocess_omics, integrate_multiomics
from tda_utils import compute_persistence_diagrams, compute_all_layers_dgms
from mapper_utils import auto_mapper, enrich_mapper_nodes
from bio_enrichment import (
    extract_cycle_genes,
    run_enrichment,
    enrich_mapper_node,
    cross_reference_genage,
    plot_enrichment_heatmap,
)
from aging_scores import phenoage_simplified, assign_acceleration_group
from config import RANDOM_SEED

sns.set_style('whitegrid')
%matplotlib inline
print('✅ Imports OK')

## 1. Load & Prepare Data

Generate synthetic multi-omics data with realistic gene names and known aging signal.

In [ ]:
# Generate data with 200 transcriptomic features (named like real genes)
ds = generate_synthetic_multimics(
    n_samples=300, topology_type='circle', noise=0.06, n_features=200
)

# Assign aging groups via PhenoAge proxy
metadata = ds['metadata'].copy()
metadata['sex'] = np.random.binomial(1, 0.5, 300)

# Simulate blood biomarkers for PhenoAge
rng = np.random.default_rng(42)
metadata['albumin'] = np.clip(rng.normal(4.2, 0.35, 300), 3.0, 5.5)
metadata['creatinine'] = np.clip(rng.normal(0.9, 0.25, 300), 0.4, 2.0)
metadata['glucose'] = np.clip(rng.normal(95, 15, 300), 60, 200)
metadata['crp'] = np.clip(rng.lognormal(0.1, 0.8, 300), 0.01, 10)
metadata['lymph'] = np.clip(rng.normal(30, 8, 300), 10, 60)
metadata['mcv'] = np.clip(rng.normal(90, 5, 300), 75, 105)
metadata['rdw'] = np.clip(rng.normal(13.5, 1.2, 300), 11, 18)
metadata['alp'] = np.clip(rng.normal(70, 20, 300), 30, 150)
metadata['wbc'] = np.clip(rng.lognormal(1.8, 0.3, 300) * 1000, 3000, 12000)

metadata = assign_acceleration_group(metadata)
print(f"Groups: {metadata['aging_group'].value_counts().to_dict()}")

## 2. Extract Cycle-Associated Genes from H1 Persistence

For each persistent H1 feature (loop), identify which genes contribute most to the cycle's generator simplex.

In [ ]:
# Preprocess transcriptomics and compute persistence
data = preprocess_omics(
    pd.DataFrame(ds['transcriptomics']),
    method='standard'
)
dgms = compute_persistence_diagrams(data, max_dim=2, use_cache=False)

# Extract genes associated with H1 cycles
cycle_genes = extract_cycle_genes(
    dgms,
    data,
    dim=1,
    threshold=0.15
)

print(f"H1 cycles detected: {len(cycle_genes)}")
for i, genes in enumerate(cycle_genes[:5]):  # show first 5
    print(f"  Cycle {i+1}: {len(genes)} genes — {genes[:5]}...")

## 3. GO/KEGG Enrichment of Cycle-Associated Genes

Run functional enrichment to check if cycle gene sets are enriched in known longevity pathways (mTOR, autophagy, sirtuins, oxidative phosphorylation, etc.).

In [ ]:
# Concatenate all cycle-associated genes
all_cycle_genes = list(set().union(*cycle_genes))
print(f"Unique cycle-associated genes: {len(all_cycle_genes)}")

# Run enrichment
enrichment_results = run_enrichment(
    all_cycle_genes,
    background_size=200,
    databases=['GO_Biological_Process', 'KEGG', 'Reactome']
)

print(f"\nTop 10 enriched pathways:")
for row in enrichment_results.head(10).itertuples():
    print(f"  {row.Term:50s}  p={row.P_value:.2e}  "
          f"genes={row.Overlap}")

In [ ]:
# Visualize
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
plot_enrichment_heatmap(enrichment_results.head(15), ax=ax)
ax.set_title('Top 15 Enriched Pathways — H1 Cycle Genes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/enrichment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Mapper Node Enrichment Analysis

Build a Mapper graph and identify nodes that are statistically enriched in resilient individuals.

In [ ]:
# Integrate all omics layers
omics_dict = {
    'transcriptomics': preprocess_omics(pd.DataFrame(ds['transcriptomics']), method='standard'),
    'metabolomics': preprocess_omics(pd.DataFrame(ds['metabolomics']), method='standard'),
    'epigenomics': preprocess_omics(pd.DataFrame(ds['epigenomics']), method='standard'),
}
integrated = integrate_multiomics(omics_dict, method='concat')

# Build Mapper graph colored by aging group
graph = auto_mapper(
    integrated,
    labels=metadata['aging_group'].values,
    n_cubes=12,
    perc_overlap=0.5,
    cluster_method='DBSCAN'
)

print(f"Mapper graph: {len(graph.get('nodes', {}))} nodes, "
      f"{len(graph.get('links', []))} edges")

In [ ]:
# Identify nodes enriched in resilient individuals
enrichment = enrich_mapper_nodes(graph, metadata['aging_group'].values, group_col='aging_group')

resilient_nodes = enrichment[
    enrichment['dominant_group'] == 'resilient'
].sort_values('fisher_pvalue')

print(f"Nodes enriched in RESILIENT individuals: {len(resilient_nodes)}")
display(resilient_nodes.head(10))

# For each resilient node, find the associated features
for node_id in resilient_nodes.head(3).index:
    node_data = enrich_mapper_node(graph, node_id, integrated)
    print(f"\nNode {node_id}: {node_data['n_individuals']} individuals")
    print(f"  Top features: {node_data['top_features'][:5]}")

## 5. Cross-Reference with Longevity Databases

Compare our cycle-associated genes and resilient-node genes with known longevity genes from GenAge and LongevityMap.

In [ ]:
# Cross-reference with GenAge (built-in list in bio_enrichment.py)
genage_overlap = cross_reference_genage(all_cycle_genes)

print(f"Cycle genes overlapping with GenAge: {len(genage_overlap)}")
if len(genage_overlap) > 0:
    print(f"  Overlapping genes: {genage_overlap}")
else:
    print("  No overlap — cycle genes are distinct from known longevity genes.")
    print("  This suggests the TDA approach identifies novel signatures.")

# Fisher's exact test for significance
from scipy.stats import fisher_exact
n_cycle = len(all_cycle_genes)
n_genage = len(genage_overlap) if isinstance(genage_overlap, list) else 0
n_total = 200  # total genes in dataset
n_genage_total = 307  # GenAge human genes

table = [
    [n_genage, n_cycle - n_genage],
    [n_genage_total - n_genage, n_total - n_cycle - n_genage_total + n_genage]
]
odds_ratio, p_value = fisher_exact(table) if n_genage > 0 else (0, 1)
print(f"\nFisher's exact test:")
print(f"  Odds ratio: {odds_ratio:.2f}")
print(f"  p-value: {p_value:.4f}")

## 6. Subnetwork Visualization

Visualize the co-expression network of cycle-associated genes, highlighting those that overlap with known longevity pathways.

In [ ]:
import networkx as nx

# Build co-expression network from cycle genes
cycle_indices = [i for i in range(200) if i in all_cycle_genes or 
                 f"gene_{i}" in str(all_cycle_genes)]

# Simplified: use correlation matrix of the transcriptomic data
corr_matrix = np.corrcoef(data[:100, :].T)  # first 100 samples
threshold = 0.7

G = nx.Graph()
for i in range(min(200, corr_matrix.shape[0])):
    for j in range(i + 1, min(200, corr_matrix.shape[1])):
        if abs(corr_matrix[i, j]) > threshold:
            G.add_edge(f"G_{i}", f"G_{j}", weight=abs(corr_matrix[i, j]))

print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Highlight cycle genes
cycle_gene_set = set(cycle_indices[:20])  # top 20
node_colors = ['#e74c3c' if int(n.split('_')[1]) in cycle_gene_set 
               else '#3498db' for n in G.nodes()]

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
pos = nx.spring_layout(G, k=0.5, seed=42)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=50, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.1, ax=ax)
ax.set_title('Co-Expression Network — Red = Cycle-Associated Genes', 
             fontsize=12, fontweight='bold')
ax.axis('off')
plt.savefig('../results/figures/cycle_network.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Comparison: Accelerated vs Resilient Feature Profiles

For each Mapper node, compare the feature profiles between accelerated and resilient individuals.

In [ ]:
# Identify the top enriched resilient node
top_resilient_node = resilient_nodes.iloc[0].name if len(resilient_nodes) > 0 else None

if top_resilient_node is not None:
    # Get individuals in this node
    node_members = graph['nodes'][top_resilient_node]
    
    resilient_in_node = metadata.loc[node_members][
        metadata.loc[node_members, 'aging_group'] == 'resilient'
    ].index
    accel_in_node = metadata.loc[node_members][
        metadata.loc[node_members, 'aging_group'] == 'accelerated'
    ].index
    
    print(f"Node {top_resilient_node}:")
    print(f"  Resilient: {len(resilient_in_node)}")
    print(f"  Accelerated: {len(accel_in_node)}")
    
    # Differential feature analysis
    if len(resilient_in_node) > 3 and len(accel_in_node) > 3:
        from scipy.stats import ttest_ind
        
        resilient_data = data[resilient_in_node]
        accel_data = data[accel_in_node]
        
        t_stats, p_values = ttest_ind(resilient_data, accel_data, axis=0)
        
        diff_df = pd.DataFrame({
            'feature': [f'G_{i}' for i in range(len(t_stats))],
            't_statistic': t_stats,
            'p_value': p_values,
            'abs_t': np.abs(t_stats)
        }).sort_values('abs_t', ascending=False)
        
        print(f"\nTop 10 differential features (resilient vs accelerated):")
        print(diff_df.head(10).to_string(index=False))

## 8. Summary & Biological Interpretation

### Key Findings

1. **H1 Cycle Genes:** The persistent H1 features identified [N] genes forming cyclical co-expression patterns.
2. **Pathway Enrichment:** Cycle-associated genes are enriched in [list top pathways].
3. **Resilient Mapper Nodes:** [N] nodes in the Mapper graph are significantly enriched in resilient individuals.
4. **Longevity Database Overlap:** [N] genes overlap with GenAge/LongevityMap [p-value].

### Biological Interpretation

- **Cyclical co-expression** in resilient individuals may indicate preserved feedback loops in key longevity pathways (mTOR, circadian rhythm).
- **Mapper node enrichment** identifies multi-omics subspaces where resilience is concentrated — potential biomarker panels.
- **Novel signatures** not in existing databases suggest TDA captures emergent properties invisible to conventional differential expression.

### Caveats

- Synthetic data — results require validation on real multi-omics datasets.
- Correlation ≠ causation — topological cycles are mathematical structures, not proven mechanisms.
- Small sample sizes may produce unstable diagrams — bootstrap validation recommended.

### Next Steps

- Validate on real InCHIANTI or ELSA data (see `scripts/download_data.py`)
- Compare with epigenetic clock features (Horvath, GrimAge)
- Integrate Santos-Pujol supercentenarian comparison (see `data_utils.compare_with_santos_pujol`)